# 📖 Wikimedia Page Views & Trending Topics — Apache Flink + Cassandra + Snowflake
**Source:** Wikimedia REST API (free, no key — api.wikimedia.org)
**Stack:** Wikimedia API → Kafka → Apache Flink → Cassandra (speed) → Snowflake (batch)
**Pattern:** Lambda Architecture (Speed + Batch)
**Orchestration:** Apache Airflow

## Overview
Real-time pipeline tracking Wikipedia article page views and trending topics
across multiple languages. Uses Apache Flink for sliding-window trending detection,
Cassandra for low-latency trending topic reads, and Snowflake for analytical
deep-dives on topic trends over time.

**APIs used:**
1. Wikimedia Pageviews API — hourly views per article per language
2. Wikimedia Recent Changes Stream (SSE) — live edit events
3. Wikimedia Featured Content API — trending/featured articles

**Free:** No API key. Rate limit: 200 req/s. Politeness: set User-Agent header.

## Section 1 — Imports & Configuration

In [ ]:
# pip install requests kafka-python apache-flink cassandra-driver snowflake-connector-python sseclient-py pandas

import requests
import json
import time
import logging
import sseclient
import pandas as pd
from datetime import datetime, timezone, timedelta, date
from kafka import KafkaProducer
from cassandra.cluster import Cluster
from cassandra.query import BatchStatement
from cassandra import ConsistencyLevel
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('WikimediaPipeline')
print('✅ All imports loaded')

In [ ]:
# Wikimedia API — free, no key
WM_BASE_URL        = 'https://api.wikimedia.org'
WM_PAGEVIEWS_URL   = 'https://wikimedia.org/api/rest_v1/metrics/pageviews'
WM_CHANGES_URL     = 'https://stream.wikimedia.org/v2/stream/recentchange'
WM_USER_AGENT      = 'DataEngineeringPortfolio/1.0 (zulham.va@gmail.com)'

# Languages to track
LANGUAGES = ['en', 'id', 'de', 'fr', 'ja', 'es', 'pt', 'ru', 'zh', 'ar']

# Top articles per language to track
TOP_N_ARTICLES = 50

# Kafka
KAFKA_BOOTSTRAP       = 'localhost:9092'
KAFKA_TOPIC_PAGEVIEWS = 'wikimedia-pageviews'
KAFKA_TOPIC_CHANGES   = 'wikimedia-recentchanges'

# Cassandra (speed layer)
CASSANDRA_HOSTS    = ['localhost']
CASSANDRA_KEYSPACE = 'wikimedia_trending'

# Snowflake (batch layer)
SNOWFLAKE_CONFIG = {
    'account':   'your_account.ap-southeast-1',
    'user':      'your_username',
    'password':  'your_password',
    'warehouse': 'COMPUTE_WH',
    'database':  'WIKIMEDIA_DW',
    'schema':    'ANALYTICS',
}

print(f'✅ Config — tracking {len(LANGUAGES)} languages, top {TOP_N_ARTICLES} articles each')

## Section 2 — Wikimedia Pageviews Producer

In [ ]:
def fetch_top_articles(lang: str, year: int, month: int, day: int) -> list:
    """
    Fetch top 50 most-viewed Wikipedia articles for a language on a given date.
    Wikimedia Pageviews API — free, no key.
    """
    url = (
        f'{WM_PAGEVIEWS_URL}/top/{lang}.wikipedia/all-access'
        f'/{year}/{month:02d}/{day:02d}'
    )
    headers = {'User-Agent': WM_USER_AGENT, 'Accept': 'application/json'}
    resp = requests.get(url, headers=headers, timeout=20)
    if resp.status_code == 404:
        return []
    resp.raise_for_status()
    data = resp.json()
    articles = data.get('items', [{}])[0].get('articles', [])
    now = datetime.now(timezone.utc).isoformat()
    records = []
    for art in articles[:TOP_N_ARTICLES]:
        title = art.get('article', '')
        # Skip special pages
        if title.startswith('Special:') or title.startswith('Wikipedia:'):
            continue
        records.append({
            'article_id':     f"{lang}:{title.replace(' ','_')}",
            'language':       lang,
            'article_title':  title,
            'article_url':    f'https://{lang}.wikipedia.org/wiki/{title.replace(" ","_")}',
            'view_date':      f'{year}-{month:02d}-{day:02d}',
            'rank':           art.get('rank', 0),
            'page_views':     int(art.get('views', 0)),
            'ingested_at':    now,
            'source':         'wikimedia-pageviews'
        })
    return records

def fetch_article_hourly_views(lang: str, title: str, hours_back: int = 24) -> list:
    """
    Fetch hourly page view counts for a specific article.
    Used to track trending velocity.
    """
    now   = datetime.now(timezone.utc)
    start = (now - timedelta(hours=hours_back)).strftime('%Y%m%d%H')
    end   = now.strftime('%Y%m%d%H')
    url   = (
        f'{WM_PAGEVIEWS_URL}/per-article/{lang}.wikipedia/all-access/all-agents'
        f'/{title}/hourly/{start}/{end}'
    )
    headers = {'User-Agent': WM_USER_AGENT}
    try:
        resp = requests.get(url, headers=headers, timeout=15)
        resp.raise_for_status()
        items = resp.json().get('items', [])
        records = []
        for item in items:
            ts_str = str(item.get('timestamp', '0000000000'))
            records.append({
                'article_id':   f'{lang}:{title}',
                'language':     lang,
                'article_title': title,
                'hour_bucket':  ts_str[:10],
                'page_views':   int(item.get('views', 0)),
                'ingested_at':  now.isoformat(),
                'source':       'wikimedia-hourly'
            })
        return records
    except Exception as e:
        logger.warning(f'Hourly views {lang}:{title}: {e}')
        return []

def stream_pageviews_to_kafka(execution_date: str) -> int:
    """
    Fetch top articles for all languages and publish to Kafka.
    """
    dt   = datetime.strptime(execution_date, '%Y-%m-%d')
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all', compression_type='gzip'
    )
    total = 0
    for lang in LANGUAGES:
        try:
            records = fetch_top_articles(lang, dt.year, dt.month, dt.day)
            for r in records:
                producer.send(
                    KAFKA_TOPIC_PAGEVIEWS,
                    key=r['article_id'].encode('utf-8'),
                    value=r
                )
                total += 1
            logger.info(f'  ✅ {lang}: {len(records)} articles published')
            time.sleep(0.5)
        except Exception as e:
            logger.warning(f'  ⚠️  {lang}: {e}')

    producer.flush()
    producer.close()
    logger.info(f'✅ Total pageview records published: {total}')
    return total

# stream_pageviews_to_kafka(str(date.today()))

## Section 3 — Wikimedia Recent Changes SSE Stream

In [ ]:
def stream_recent_changes_to_kafka(duration_seconds: int = 300) -> int:
    """
    Connect to Wikimedia's SSE (Server-Sent Events) recent changes stream.
    This is a TRUE real-time stream — events are pushed as they happen.
    Filtered to tracked languages only.
    """
    headers = {'User-Agent': WM_USER_AGENT, 'Accept': 'text/event-stream'}
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all'
    )
    lang_set = set(f'{l}.wikipedia' for l in LANGUAGES)
    published = 0
    start_time = time.time()

    logger.info(f'🌊 Connecting to Wikimedia SSE stream for {duration_seconds}s...')
    try:
        resp = requests.get(WM_CHANGES_URL, headers=headers, stream=True, timeout=60)
        client = sseclient.SSEClient(resp)
        for event in client.events():
            if time.time() - start_time > duration_seconds:
                break
            try:
                data = json.loads(event.data)
                # Filter: only tracked languages, only article edits
                if data.get('wiki') not in lang_set:
                    continue
                if data.get('type') != 'edit':
                    continue
                if data.get('namespace', -1) != 0:  # main article namespace
                    continue
                record = {
                    'change_id':      str(data.get('id', '')),
                    'wiki':           data.get('wiki', ''),
                    'language':       data.get('wiki', '').replace('.wikipedia', ''),
                    'article_title':  data.get('title', ''),
                    'change_type':    data.get('type', ''),
                    'revision_new':   data.get('revision', {}).get('new', 0),
                    'revision_old':   data.get('revision', {}).get('old', 0),
                    'byte_delta':     int(data.get('length', {}).get('new', 0)) -
                                      int(data.get('length', {}).get('old', 0)),
                    'is_bot':         data.get('bot', False),
                    'is_minor':       data.get('minor', False),
                    'is_new':         data.get('type') == 'new',
                    'user':           data.get('user', ''),
                    'event_time':     datetime.fromtimestamp(
                                        data.get('timestamp', 0), tz=timezone.utc
                                      ).isoformat(),
                    'comment':        (data.get('comment', ''))[:200],
                    'ingested_at':    datetime.now(timezone.utc).isoformat(),
                    'source':         'wikimedia-sse-stream'
                }
                producer.send(KAFKA_TOPIC_CHANGES, key=record['change_id'].encode(), value=record)
                published += 1
                if published % 100 == 0:
                    logger.info(f'  📡 {published} change events published...')
            except json.JSONDecodeError:
                continue
    except Exception as e:
        logger.error(f'SSE stream error: {e}')
    finally:
        producer.flush()
        producer.close()

    logger.info(f'✅ SSE stream done: {published} change events published')
    return published

# stream_recent_changes_to_kafka(duration_seconds=60)

## Section 4 — Flink Trending Detection + Cassandra Sink

In [ ]:
FLINK_TRENDING_JOB = '''
# Flink SQL job for sliding-window trending detection
# Identifies articles with rapidly increasing edit velocity

t_env.execute_sql("""
    CREATE TABLE kafka_changes (
        change_id      STRING,
        language       STRING,
        article_title  STRING,
        byte_delta     INT,
        is_bot         BOOLEAN,
        is_minor       BOOLEAN,
        event_time     TIMESTAMP(3),
        WATERMARK FOR event_time AS event_time - INTERVAL 30 SECOND
    ) WITH (
        'connector' = 'kafka',
        'topic'     = 'wikimedia-recentchanges',
        'properties.bootstrap.servers' = 'localhost:9092',
        'format'    = 'json',
        'scan.startup.mode' = 'latest-offset'
    )
""")

# 10-minute sliding window trending score
# High edit velocity (esp non-bot) = trending article
t_env.execute_sql("""
    SELECT
        window_start, window_end,
        language, article_title,
        COUNT(*)                                   AS edit_count,
        SUM(CASE WHEN NOT is_bot THEN 1 ELSE 0 END) AS human_edit_count,
        SUM(ABS(byte_delta))                        AS total_byte_change,
        -- Trending score: human edits weighted more than bot edits
        SUM(CASE WHEN NOT is_bot THEN 3 ELSE 1 END) AS trending_score
    FROM TABLE(
        TUMBLE(TABLE kafka_changes, DESCRIPTOR(event_time), INTERVAL 10 MINUTE)
    )
    WHERE NOT is_minor  -- exclude minor edits from trending
    GROUP BY TUMBLE(event_time, INTERVAL 10 MINUTE), language, article_title
    HAVING COUNT(*) >= 3  -- at least 3 edits in 10 minutes
""")
'''

# Cassandra schema for trending topics
CASSANDRA_TRENDING_DDL = """
CREATE KEYSPACE IF NOT EXISTS wikimedia_trending
WITH REPLICATION = {'class': 'SimpleStrategy', 'replication_factor': 1};

-- Speed layer: top trending per language, updated every 10 minutes
CREATE TABLE IF NOT EXISTS wikimedia_trending.trending_articles (
    language          TEXT,
    window_start      TIMESTAMP,
    article_title     TEXT,
    edit_count        INT,
    human_edit_count  INT,
    total_byte_change INT,
    trending_score    INT,
    ingested_at       TEXT,
    PRIMARY KEY (language, window_start, trending_score)
) WITH CLUSTERING ORDER BY (window_start DESC, trending_score DESC)
  AND default_time_to_live = 86400;  -- 24-hour TTL

-- Pageviews time series
CREATE TABLE IF NOT EXISTS wikimedia_trending.article_pageviews (
    language      TEXT,
    view_date     TEXT,
    article_title TEXT,
    page_views    INT,
    rank          INT,
    PRIMARY KEY (language, view_date, article_title)
) WITH CLUSTERING ORDER BY (view_date DESC);
"""

def write_trending_to_cassandra(records: list):
    cluster = Cluster(CASSANDRA_HOSTS)
    session = cluster.connect(CASSANDRA_KEYSPACE)
    stmt = session.prepare("""
        INSERT INTO trending_articles
        (language, window_start, article_title, edit_count,
         human_edit_count, total_byte_change, trending_score, ingested_at)
        VALUES (?,?,?,?,?,?,?,?)
        USING TTL 86400
    """)
    batch = BatchStatement(consistency_level=ConsistencyLevel.LOCAL_QUORUM)
    for r in records:
        batch.add(stmt, [
            r['language'], r['window_start'], r['article_title'],
            r['edit_count'], r['human_edit_count'],
            r['total_byte_change'], r['trending_score'],
            datetime.now(timezone.utc).isoformat()
        ])
    session.execute(batch)
    cluster.shutdown()
    logger.info(f'⚡ {len(records)} trending records → Cassandra')

# write_trending_to_cassandra([])

## Section 5 — Batch Layer: Pageviews → Snowflake

In [ ]:
def load_pageviews_to_snowflake(execution_date: str) -> int:
    """
    BATCH LAYER: Load daily top-article pageviews to Snowflake for
    trend analysis, cross-language comparison, topic classification.
    """
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import from_json, col, lag, round as spark_round
    from pyspark.sql.types import StructType, StructField, StringType, IntegerType
    from pyspark.sql.window import Window

    spark = SparkSession.builder.appName('WikiBatch').getOrCreate()
    schema = StructType([
        StructField('article_id',    StringType(),  True),
        StructField('language',      StringType(),  True),
        StructField('article_title', StringType(),  True),
        StructField('view_date',     StringType(),  True),
        StructField('rank',          IntegerType(), True),
        StructField('page_views',    IntegerType(), True),
        StructField('ingested_at',   StringType(),  True),
    ])
    raw = spark.read.format('kafka') \
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
        .option('subscribe', KAFKA_TOPIC_PAGEVIEWS) \
        .option('startingOffsets', 'earliest') \
        .option('endingOffsets', 'latest') \
        .load()
    df = raw.select(from_json(col('value').cast('string'), schema).alias('d')).select('d.*') \
        .dropDuplicates(['article_id', 'view_date'])
    w = Window.partitionBy('language', 'article_title').orderBy('view_date')
    df_enriched = df \
        .withColumn('prev_views',  lag('page_views', 1).over(w)) \
        .withColumn('views_yoy_change_pct',
            spark_round((col('page_views') - col('prev_views')) / col('prev_views') * 100, 2)
        ) \
        .drop('prev_views')
    count = df_enriched.count()
    pdf = df_enriched.toPandas()
    pdf.columns = [c.upper() for c in pdf.columns]
    conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
    conn.cursor().execute(
        'DELETE FROM WIKIMEDIA_DW.ANALYTICS.ARTICLE_PAGEVIEWS WHERE VIEW_DATE = %s',
        (execution_date,)
    )
    write_pandas(conn, pdf, 'ARTICLE_PAGEVIEWS', database='WIKIMEDIA_DW', schema='ANALYTICS')
    conn.close()
    spark.stop()
    logger.info(f'✅ {count} pageview records → Snowflake')
    return count

# load_pageviews_to_snowflake(str(date.today()))

## Section 6 — Pipeline Summary

| Layer | Technology | Details |
|---|---|---|
| **Source 1** | Wikimedia Pageviews API | Top 50 articles × 10 languages × daily |
| **Source 2** | Wikimedia SSE Stream | Live edit events, true real-time push |
| **Ingest** | Apache Kafka | 2 topics: pageviews + recentchanges |
| **Stream** | Apache Flink | 10-min tumbling window trending score |
| **Speed Store** | Cassandra | Trending topics, TTL 24h, ms latency |
| **Batch Store** | Snowflake | Historical pageviews, cross-language analytics |
| **Orchestration** | Airflow | 2 DAGs: SSE stream (hourly) + batch (daily) |

**Trending score formula:**
`score = (human_edits × 3) + (bot_edits × 1)` in 10-minute window
High score = article being actively edited = likely breaking news or trending topic

**SSE (Server-Sent Events):** Unlike polling, Wikimedia SSE pushes changes instantly.
This is a true event-driven stream — latency < 1 second from edit to Kafka publish.